In [1]:
"""HipMRI dataset discovery and patient split checks."""

from pathlib import Path
import re


# Extract the patient ID while checking the expected filename structure.
IMAGE_PATTERN = re.compile(
    r"case_(\d+)_week_\d+_slice_\d+\.nii\.gz"
)
SPLITS = ("train", "validate", "test")


def find_pairs(data_dir, split):
    """Return matching image and mask paths for one supplied split."""
    if split not in SPLITS:
        raise ValueError(f"Unknown split: {split}")

    data_dir = Path(data_dir)
    image_dir = data_dir / f"keras_slices_{split}"
    mask_dir = data_dir / f"keras_slices_seg_{split}"

    # Catch incorrect paths before attempting to load any images.
    for folder in (image_dir, mask_dir):
        if not folder.is_dir():
            raise FileNotFoundError(f"Dataset folder not found: {folder}")

    images = sorted(image_dir.glob("*.nii.gz"))
    if not images:
        raise ValueError(f"No NIfTI images found in {image_dir}")

    pairs = []
    expected_masks = set()

    for image_path in images:
        if IMAGE_PATTERN.fullmatch(image_path.name) is None:
            raise ValueError(f"Unexpected image filename: {image_path.name}")

        # Pair by filename rather than relying on directory ordering.
        mask_name = image_path.name.replace("case_", "seg_", 1)
        mask_path = mask_dir / mask_name

        if not mask_path.is_file():
            raise FileNotFoundError(f"Missing mask: {mask_path}")

        pairs.append((image_path, mask_path))
        expected_masks.add(mask_name)

    # Also detect masks that have no corresponding image.
    actual_masks = {p.name for p in mask_dir.glob("*.nii.gz")}
    extra_masks = actual_masks - expected_masks
    if extra_masks:
        raise ValueError(
            f"{split}: {len(extra_masks)} masks have no matching image"
        )

    return pairs


def patient_ids(pairs):
    """Read patient IDs from the verified image filenames."""
    return {
        IMAGE_PATTERN.fullmatch(image.name).group(1)
        for image, _ in pairs
    }


def audit_splits(data_dir):
    """Check all supplied splits and reject overlapping patient IDs."""
    splits = {split: find_pairs(data_dir, split) for split in SPLITS}
    patients = {
        split: patient_ids(pairs) for split, pairs in splits.items()
    }

    for first, second in (
        ("train", "validate"),
        ("train", "test"),
        ("validate", "test"),
    ):
        overlap = patients[first] & patients[second]
        if overlap:
            raise ValueError(
                f"Patient overlap between {first} and {second}: "
                f"{sorted(overlap)}"
            )

    print(f"{'Split':<12} {'Pairs':>8} {'Patients':>10}")
    for split in SPLITS:
        print(
            f"{split:<12} {len(splits[split]):>8} "
            f"{len(patients[split]):>10}"
        )

    print("All image–mask pairs matched; no patient overlap.")
    return splits


# Use the dataset's absolute location while developing in a notebook.
data_dir = Path(
    "/lambda/nfs/comp3710-virginia/PatternAnalysis-2026/"
    "recognition/ResidualUNet_s4749642/data/keras_slices_data"
)

splits = audit_splits(data_dir)

Split           Pairs   Patients
train           11460         31
validate          660          4
test              540          3
All image–mask pairs matched; no patient overlap.


In [2]:
# %pip install nibabel

In [3]:
import nibabel as nib
import numpy as np

# Sanity test - make sure all of the image pairs actually work
def load_pair(image_path, mask_path):
    """Load and validate one 2D MRI–mask pair."""
    image_nii = nib.load(image_path)
    mask_nii = nib.load(mask_path)

    # Remove singleton dimensions, keeping the stored orientation.
    image = np.squeeze(image_nii.get_fdata(dtype=np.float32))
    mask = np.squeeze(mask_nii.get_fdata())

    if image.ndim != 2 or mask.ndim != 2:
        raise ValueError("Expected 2D image and mask")

    if image.shape != mask.shape:
        raise ValueError("Image and mask dimensions differ")

    # Check spatial metadata and numerical validity before preprocessing.
    if not np.allclose(image_nii.affine, mask_nii.affine):
        raise ValueError("Image and mask affines differ")

    if not np.isfinite(image).all() or not np.isfinite(mask).all():
        raise ValueError("Image or mask contains NaN or infinite values")

    if not np.allclose(mask, np.round(mask)) or mask.min() < 0:
        raise ValueError("Mask must contain non-negative integer labels")

    # Standardise MRI intensities; leave segmentation labels unchanged.
    image = (image - image.mean()) / (image.std() + 1e-8)
    mask = np.round(mask).astype(np.int64)

    return image, mask

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [4]:
# Load the first verified training pair.
image_path, mask_path = splits["train"][0]
image, mask = load_pair(image_path, mask_path)

print("Example:", image_path.name)
print("Image:", image.shape, image.dtype)
print("Mask:", mask.shape, mask.dtype)
print("Normalised image mean:", round(float(image.mean()), 4))
print("Normalised image standard deviation:", round(float(image.std()), 4))
print("Mask labels:", np.unique(mask))

Example: case_004_week_0_slice_0.nii.gz
Image: (256, 128) float32
Mask: (256, 128) int64
Normalised image mean: 0.0
Normalised image standard deviation: 1.0
Mask labels: [0 1 2 3]


In [5]:
# Search the dataset and its parent folder for label documentation.
extensions = {".txt", ".md", ".json", ".csv", ".yaml", ".yml", ".xml"}
documents = []

for path in data_dir.parent.rglob("*"):
    if path.is_file() and (
        path.suffix.lower() in extensions
        or "readme" in path.name.lower()
    ):
        documents.append(path)

print("Documentation files:")
for path in sorted(documents):
    print(path.relative_to(data_dir.parent))

if not documents:
    print("No documentation found beside the uploaded dataset.")

Documentation files:
No documentation found beside the uploaded dataset.


In [6]:
# Some NIfTI files include descriptions or extensions with label information.
mask_nii = nib.load(splits["train"][0][1])

print("Description:", mask_nii.header["descrip"].item())
print("Auxiliary information:", mask_nii.header["aux_file"].item())
print("Extensions:", mask_nii.header.extensions)

Description: b''
Auxiliary information: b''
Extensions: Nifti1Extensions()


In [7]:
import torch
from torch.utils.data import Dataset, DataLoader


class HipMRIDataset(Dataset):
    """Load MRI slices and their corresponding integer-label masks."""

    def __init__(self, pairs):
        self.pairs = list(pairs)

    def __len__(self):
        # The DataLoader uses this to determine the dataset size.
        return len(self.pairs)

    def __getitem__(self, index):
        image_path, mask_path = self.pairs[index]
        image, mask = load_pair(image_path, mask_path)

        # Add the single MRI channel: [height, width] → [1, height, width].
        image = torch.from_numpy(
            np.ascontiguousarray(image)
        ).unsqueeze(0)

        # Preserve integer label IDs for CrossEntropyLoss.
        mask = torch.from_numpy(np.ascontiguousarray(mask))

        return image, mask

In [8]:
train_dataset = HipMRIDataset(splits["train"])

# Start with zero worker processes for straightforward notebook debugging.
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0,
)

images, masks = next(iter(train_loader))

print("Training slices:", len(train_dataset))
print("Image batch:", tuple(images.shape), images.dtype)
print("Mask batch:", tuple(masks.shape), masks.dtype)
print("Labels in this batch:", torch.unique(masks).tolist())

Training slices: 11460
Image batch: (16, 1, 256, 128) torch.float32
Mask batch: (16, 256, 128) torch.int64
Labels in this batch: [0, 1, 2, 3, 4, 5]
